# Notebook 6: Counting Soho District Building Footprints in Voronoi Cells

**Atlas source reading copy · local-server**

[Course guide](index.ipynb) · [Pinned original](https://github.com/PHI-Case-Studies/1854-Cholera-Outbreak-London-Advanced-2/blob/d1b56c135b0fab2f470b837a51e6777dfc20b286/NB05-Buildings-in-Voronoi-Cells.ipynb)



Use the source local/server environment for native GIS, shell, installed project or legacy dependencies; Lab provides a reading/editing copy. Follow notebook order to regenerate omitted outputs; optional unlicensed media are external continuations.

**Provenance:** `NB05-Buildings-in-Voronoi-Cells.ipynb` at `d1b56c135b0fab2f470b837a51e6777dfc20b286`; original SHA-256 `32bbd201c52871cf00eb86814fa57ca03edc23fdfc158ea7d9ad4b585ecbf7b5`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


In [ ]:
%run resources/library.py

# Notebook 6: Counting Soho District Building Footprints in Voronoi Cells

In [ ]:
import osmnx as ox

ox.__version__

## Using OSMNx `footprints` feature

Let's assume that most buildings in 1854 Soho district are still intact and have footprints accessible through OSMNx. Let's use the SOHO_COORDINATES from the Cholera Basic Notebooks as a starting point for retrieving building footprints (polygons) using OSMNx `footprints` feature.

In [ ]:
SOHO_COORDINATES = (51.513578, -0.136722)

In [ ]:
# radius of 700m
distance = 700
footprints_gdf = \
   footprints_gdf = \
   ox.features_from_point(SOHO_COORDINATES, dist=distance, \
                tags={'building':True}).to_crs(epsg=3395)

In [ ]:
footprints_gdf.crs

Let's look at the `geometry` column - these are the building footprint polygons.

In [ ]:
footprints_gdf = footprints_gdf[footprints_gdf['geometry'].notnull()]

footprints_gdf.crs

In [ ]:
footprints_gdf['geometry'].head()

In [ ]:
footprints_gdf['geometry'].plot(figsize=(14,12), color='orange')

In [ ]:
footprints_gdf['geometry'].centroid

In [ ]:
import geopandas as gpd

gpd.__version__

## Creating a geodataframe of footprint centroids

The footprint feature of OSMNx allows you to plot polygons of buildings. But to determine if buildings are inside (belongs to) pump voronoi cells, we need to identify buildings by the centroid of its polygon vertices. Fortunately, `geometry` columns in Geopandas are native `shapely` objects. We can get polygon centroids by the attribute `centroid`.

In [ ]:
footprints_centroid_gdf = gpd.GeoDataFrame(\
                            geometry=footprints_gdf['geometry'].centroid)\
                            .reset_index()\
                            .rename(columns={'index':'osmid'})

footprints_centroid_gdf.info()

In [ ]:
footprints_centroid_gdf.crs

In [ ]:
footprints_centroid_gdf.geometry.plot(markersize=4)

## Amplified cognition: Geopandas plots and computational geometry tricks

Because of of the different kinds of data that we already have, what if we could draw a Geopandas plot that looks like this?

[Source image continuation: ](https://github.com/PHI-Case-Studies/1854-Cholera-Outbreak-London-Advanced-2/blob/d1b56c135b0fab2f470b837a51e6777dfc20b286/images/subplots_overlay.png)

### Inventory of geodataframes and dataframes to use

Steps to draw this plot, layer by layer (bottom to top):

1. Building footprints, as  polygons, minus footprint polygons within pump Voronoi cells
2. Building footprints, as centroids (points), minus footprint centroids outside pump Voronoi cells
3. Building footprints, centroids within pump Voronoi cells
4. Pump Voronoi cells (clipped)
5. Street pumps

Let's reload the `clipped_voronoi_gdf` geodataframe. This is what we will use for geopandas `sjoin()` and `overlay()` functions. 

In [ ]:
clipped_voronoi_gdf = gpd.read_file('outputs/clipped_voronoi.shp').to_crs(epsg=3395)

In [ ]:
clipped_voronoi_gdf

In [ ]:
clipped_voronoi_gdf.rename(\
            columns={'vorpoint_l':'vorpt_lat',\
                     'vorpoint_1':'vorpt_lon',\
                     'points_in_polygon':'pts_in_vor'
                    }, inplace=True)

In [ ]:
clipped_voronoi_gdf.geometry.to_crs('EPSG:3395')

In [ ]:
clipped_voronoi_gdf

In [ ]:
clipped_voronoi_gdf.crs

In [ ]:
footprints_gdf.crs

In [ ]:
footprints_gdf.geometry

In [ ]:
footprints_gdf = footprints_gdf[footprints_gdf['geometry'].apply(lambda x : x.type!='Point' )]

footprints_gdf.geometry

In [ ]:
clipped_voronoi_gdf.geometry

### Geometry intersection using `overlay()`

We will use Geopandas' `overlay` function to perform computational geometry tasks. You can go this [link](http://geopandas.org/set_operations.html) to learn more about this. The intersection operation below creates a part of the footprint landscape in the shape of the clipped Voronoi cells.

In [ ]:
intersection = gpd.overlay(footprints_gdf, clipped_voronoi_gdf, how='intersection', keep_geom_type=True, make_valid=True)

intersection.plot()

### Geometry difference using `overlay()`

The performance of the `difference` function results in a plot of the building footprints minus the area that falls under the clipped Voronoi cells.

In [ ]:
difference1 = gpd.overlay(footprints_gdf, clipped_voronoi_gdf, how='difference')

difference1.plot()

### Point geometry within polygon geometry using Geopandas `sjoin()`

The next difference operation is between geometries for building centroids (`footprints_centroid_gdf`, `shapely` points) and the clipped Voronoi polygons (`clipped_voronoi_gdf`, `shapely` polygons), and is an `sjoin` geopandas operation. The result is a subset of building footprint centroids that lie within the polygons.

In [ ]:
difference2 = \
    gpd.sjoin(footprints_centroid_gdf, clipped_voronoi_gdf, how="inner", predicate='intersects')

In [ ]:
difference2

In [ ]:
difference2['geometry'].plot(markersize=4)

In [ ]:
clipped_voronoi_gdf.plot(facecolor="none", edgecolor='blue',linewidth=2)

In [ ]:
pumps_gdf = gpd.read_file('outputs/pumps.shp').to_crs(epsg=3395)

pumps_gdf.plot(color='black')

## Summary of plots

As an exercise in Pandas plots, let's put together all the plots above. The numbers "331" in the function `add_subplot(331)` mean (rows, columns, graph position or order). `(331)` means "draw me subplots of 3 rows by 3 columns, use position 1".

In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline

Let's review all the subplots before we overlay them togther in one plot.

In [ ]:
fig = plt.figure(figsize=(15,15))

ax1 = fig.add_subplot(331)
# make all the other subplots share x and y scales with ax1
ax2 = fig.add_subplot(332, sharey=ax1, sharex=ax1)
ax3 = fig.add_subplot(333, sharey=ax1, sharex=ax1)
ax4 = fig.add_subplot(334, sharey=ax1, sharex=ax1)
ax5 = fig.add_subplot(335, sharey=ax1, sharex=ax1)
ax6 = fig.add_subplot(336, sharey=ax1, sharex=ax1)
ax7 = fig.add_subplot(337, sharey=ax1, sharex=ax1)

fs = 14
footprints_gdf['geometry'].plot(color='gray', alpha=0.2, ax=ax1)
ax1.set_title('Layer 1: Building Polygons from OSMNx', fontsize=fs)

footprints_centroid_gdf.geometry.plot(color='black', alpha=0.2, markersize=1, ax=ax2)
ax2.set_title('Layer 2: Building Polygon Centroids', fontsize=fs)

difference1.plot(color='gray', alpha=0.2, ax=ax3)
ax3.set_title('Layer 3: Polygons Outside Voronoi Boundaries', fontsize=fs)

intersection.plot(color='orange', alpha=0.3, ax=ax4)
ax4.set_title('Layer 4: Polygons Inside Voronoi Boundaries', fontsize=fs)

difference2['geometry'].plot(color='red', alpha=0.3, markersize=1, ax=ax5)
ax5.set_title('Layer 5: Centroids Inside Voronoi Boundaries', fontsize=fs)

clipped_voronoi_gdf.plot(facecolor="none", edgecolor='blue', linewidth=0.5, ax=ax6)
ax6.set_title('Layer 6: Clipped Pumped Voronoi Polygons', fontsize=fs)

pumps_gdf.plot(color='black', ax=ax7, markersize=3)
ax7.set_title('Layer 7: Street Pumps', fontsize=fs)

plt.tight_layout()
plt.show()

### Put all the subplots together, layer by layer

Let's now add the subplots layer by layer to create a final plot.

In [ ]:
fig, ax = plt.subplots(figsize=(15,15))

# BUILDING FOOTPRINTS POLYGONS LAYER
# layer 1: building footprint polygons minus footprint polygons inside Voronoi cells
difference1['geometry'].plot(color='gray', ax=ax, alpha=0.2)
# layer 2: building footprint polygons that fit within Voronoi cells
intersection.plot(ax=ax, color='orange', linewidth=2, alpha=0.3)

# VORONOI POLYGONS
# layer 3: Voronoi cells from pumps, no fill color
clipped_voronoi_gdf.plot(ax=ax, facecolor="none", edgecolor='blue',linewidth=2)

# BUILDING FOOTPRINTS CENTROID, POINTS LAYER
# layer 4: black points, outside Voronoi cells
footprints_centroid_gdf.geometry.plot(color='black', ax=ax, alpha=0.2, markersize=4)
# layer 5: red points, inside Voronoi cells
difference2['geometry'].plot(color='red', ax=ax, alpha=0.3, markersize=4)
pumps_gdf.plot(color='black', ax=ax)

ax.set_title('Overlay of Clipped Pump Voronoi Cells with Building Footprint Centroids', \
             fontsize=16)
ax.tick_params(axis = 'both', which = 'major', labelsize = 14)
ax.set_ylabel('Latitude', fontsize=16)
ax.set_xlabel('Longitude', fontsize=16)

Orange or gray polygons are the building footprints. Grey or red dots represent the building footprint centroids.

## TBC: Reuse the code for identifying death locations within Voronoi cells

In [ ]:
%run resources/points_in_polygons.py

In [ ]:
# Read the data.
polygons = clipped_voronoi_gdf
points = footprints_centroid_gdf

polygons = points_in_polygons(points, polygons)

polygons

What we have now is a geodataframe with the number of building centroids found in each Voronoi polygon.

In [ ]:
clipped_voronoi_buildings_gdf = \
    polygons.rename(columns={'vorpoint_l':'vorpt_lat',\
                             'vorpoint_1':'vorpt_lon',\
                            'points_in_polygon':'num_bldgs'})

clipped_voronoi_buildings_gdf

Let's save these to files.

In [ ]:
clipped_voronoi_buildings_gdf.to_file('outputs/clipped_voronoi_buildings.shp')
clipped_voronoi_buildings_gdf.to_file("outputs/clipped_voronoi_buildings_gdf.json", \
                                      driver="GeoJSON")

## Create the barchart for buildings in Voronoi cells 

In [ ]:
import pandas as pd

pd.__version__

### Buildings per street  pump Voronoi cell

In [ ]:
ax = pd.DataFrame(clipped_voronoi_buildings_gdf)\
    .plot.barh(y='num_bldgs', x='Street', color='k', figsize=(8,5))
ax.set_title('Building Centroids in Voronoi Cells of Pumps (by Street Location)', \
             fontsize=16)
ax.tick_params(axis = 'both', which = 'major', labelsize = 14)
ax.set_ylabel('Voronoi Cell by Pump', fontsize=16)
ax.set_xlabel('Building Count', \
              fontsize=16)

Which street pump catchment area has the most building footprints (assuming these have been around in 1854)?

## Merge two geodataframes as Pandas dataframes with selected columns to plot two barcharts

In [ ]:
death_locations_df = gpd.read_file('outputs/clipped_voronoi_counts.shp')\
    [['FID','points_in_','Street']].rename(columns={'points_in_':'death_locations'})

death_locations_df

In [ ]:
buildings_df = \
        clipped_voronoi_buildings_gdf[['FID','num_bldgs']]\
        .rename(columns={'num_bldgs':'buildings'})

buildings_df

In [ ]:
voronoi_deaths_buildings_df = pd.merge(death_locations_df, buildings_df, \
                                       on='FID', how='inner')

voronoi_deaths_buildings_df

### Plotting the two barcharts together...

In [ ]:
import matplotlib.pyplot as plt

ax1 = voronoi_deaths_buildings_df[['death_locations',\
                                  'buildings',\
                                  'Street']]\
    .rename(columns={'death_locations':'Death Locations',\
                    'buildings':'Buildings',\
                    'buildings per sq. meter':'Buildings per Sq. Meter'})\
    .plot.barh(figsize=(10,8), x='Street', colormap='Accent')


ax1.set_title('Building Centroids and Building Density in Street Pump Voronoi Cells', \
             fontsize=18)
ax1.tick_params(axis = 'both', which = 'major', labelsize = 14)
ax1.set_ylabel('Voronoi Cell by Pump', fontsize=16)
ax.set_xlabel('Point Count', \
              fontsize=16)


## Next Steps - Challenge

What's left to do is to plot the subplots on folium - do you think you can figure that out?

## References

1. Geopandas SJoin: http://geopandas.org/reference/geopandas.sjoin.html
2. Geopandas Overlay Set Operations: http://geopandas.org/set_operations.html
3. Pandas visualization guide: https://pandas.pydata.org/pandas-docs/version/0.25/user_guide/visualization.html
4. Matplotlib color map guide: https://matplotlib.org/examples/color/colormaps_reference.html
5. Stack Overflow reference for multiple plots, odd numbered: https://stackoverflow.com/questions/41025187/matplotlib-add-subplot-odd-number-of-plots